In [ ]:
import os
import numpy as np
import pandas as pd
from obspy.clients.fdsn import Client
from obspy import UTCDateTime
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

# ============================================================
# CONFIG
# ============================================================

KATALOG_PATH = "/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/cek_data_output/03_katalog_gabungan_berlabel.csv"
OUTPUT_DIR = "/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/waveform_irisan_usgs_bmkg_snr"

NETWORKS = ["IU", "II", "GE", "G", "IA"]
CHANNELS = ["BHZ", "BHN", "BHE"]

PREVIEW_BEFORE = 1
PREVIEW_AFTER = 2

WINDOW_BEFORE = 60
WINDOW_AFTER = 60

MAX_WORKERS = 6

client = Client("IRIS")

# ============================================================
# LOAD KATALOG IRISAN
# ============================================================

df = pd.read_csv(KATALOG_PATH)
df['time_dt'] = pd.to_datetime(df['time_dt'], errors='coerce')
iris = df[df['Kategori_Katalog'] == "Irisan (BMKG+USGS)"].copy()

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# HITUNG SNR
# ============================================================

def compute_snr(trace):
    data = trace.data.astype(float)
    if len(data) < 50:
        return 0
    noise = data[:int(0.3 * len(data))]
    signal = data[int(0.3 * len(data)):]
    snr = (np.std(signal) + 1e-6) / (np.std(noise) + 1e-6)
    return snr

# ============================================================
# PILIH STASIUN TERBAIK BERDASARKAN SNR
# ============================================================

def pick_best_station(event_id, origin):
    best_snr = -1
    best_station = None
    best_network = None

    start = origin - PREVIEW_BEFORE
    end = origin + PREVIEW_AFTER

    for net in NETWORKS:
        try:
            st = client.get_waveforms(net, "*", "*", "BHZ", start, end)
        except:
            continue

        for tr in st:
            snr = compute_snr(tr)
            if snr > best_snr:
                best_snr = snr
                best_station = tr.stats.station
                best_network = tr.stats.network

    return best_network, best_station

# ============================================================
# DOWNLOAD 3 KANAL DARI STASIUN TERBAIK
# ============================================================

def download_event(row):
    event_id = row['Event ID']
    origin = UTCDateTime(row['time_dt'])

    event_dir = os.path.join(OUTPUT_DIR, event_id)
    os.makedirs(event_dir, exist_ok=True)

    # skip jika sudah lengkap
    if all(os.path.exists(os.path.join(event_dir, f"{ch}.mseed")) for ch in CHANNELS):
        return "skip"

    net, sta = pick_best_station(event_id, origin)
    if net is None or sta is None:
        return "fail"

    start = origin - WINDOW_BEFORE
    end = origin + WINDOW_AFTER

    success = 0
    for ch in CHANNELS:
        out_file = os.path.join(event_dir, f"{ch}.mseed")
        try:
            st = client.get_waveforms(net, sta, "*", ch, start, end)
            st.write(out_file, format="MSEED")
            success += 1
        except:
            continue

    return "ok" if success == 3 else "fail"

# ============================================================
# MULTI-THREAD + PROGRESS BAR
# ============================================================

results = {"ok": 0, "fail": 0, "skip": 0}

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(download_event, row) for _, row in iris.iterrows()]

    for future in tqdm(as_completed(futures), total=len(futures), desc="Downloading SNR-best 3C Waveforms"):
        status = future.result()
        results[status] += 1

print("\n=== RINGKASAN ===")
print(f"Berhasil   : {results['ok']}")
print(f"Gagal      : {results['fail']}")
print(f"Sudah ada  : {results['skip']}")
print("=== SELESAI ===")


/opt/homebrew/Caskroom/miniforge/base/envs/mcu_quake_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/homebrew/Caskroom/miniforge/base/envs/mcu_quake_env/lib/python3.10/site-packages/obspy/clients/fdsn/client.py:251: ObsPyDeprecationWarning: IRIS is now EarthScope, please consider changing the FDSN client short URL to 'EARTHSCOPE'.
  warnings.warn(msg, ObsPyDeprecationWarning)


In [ ]:
import os
import numpy as np
import pandas as pd
from obspy.clients.fdsn import Client
from obspy import UTCDateTime
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

# ============================================================
# 1. CONFIG
# ============================================================

KATALOG_PATH = "/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/cek_data_output/03_katalog_gabungan_berlabel.csv"
OUTPUT_DIR = "/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/waveform_irisan_usgs_bmkg_snr"

YEAR_START = 2004
YEAR_END   = 2010

NETWORKS = ["IU", "II", "GE", "G", "IA"]

# fallback prefix 3-komponen
CHANNEL_PREFIX = ["BH", "HH", "EH", "SH", "LH", "VH"]

PREVIEW_BEFORE = 1
PREVIEW_AFTER = 2

WINDOW_BEFORE = 60
WINDOW_AFTER = 60

MAX_WORKERS = 20

client = Client("IRIS")

# ============================================================
# 2. LOAD KATALOG & FILTER TAHUN
# ============================================================

df = pd.read_csv(KATALOG_PATH)
df['time_dt'] = pd.to_datetime(df['time_dt'], errors='coerce')

iris = df[df['Kategori_Katalog'] == "Irisan (BMKG+USGS)"].copy()
iris = iris[(iris['time_dt'].dt.year >= YEAR_START) & 
            (iris['time_dt'].dt.year <= YEAR_END)]

print(f"Total event IRISAN {YEAR_START}-{YEAR_END}: {len(iris)}")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# 3. HITUNG SNR
# ============================================================

def compute_snr(trace):
    data = trace.data.astype(float)
    if len(data) < 50:
        return 0
    noise = data[:int(0.3 * len(data))]
    signal = data[int(0.3 * len(data)):]
    return (np.std(signal) + 1e-6) / (np.std(noise) + 1e-6)

# ============================================================
# 4. PILIH STASIUN TERBAIK (SNR)
# ============================================================

def pick_best_station(origin):
    best_snr = -1
    best_station = None
    best_network = None

    start = origin - PREVIEW_BEFORE
    end = origin + PREVIEW_AFTER

    for net in NETWORKS:
        try:
            st = client.get_waveforms(net, "*", "*", "BHZ", start, end)
        except:
            continue

        for tr in st:
            snr = compute_snr(tr)
            if snr > best_snr:
                best_snr = snr
                best_station = tr.stats.station
                best_network = tr.stats.network

    return best_network, best_station

# ============================================================
# 5. CARI PREFIX 3-KOMPONEN YANG TERSEDIA
# ============================================================

def find_available_prefix(net, sta, origin):
    start = origin - PREVIEW_BEFORE
    end = origin + PREVIEW_AFTER

    for prefix in CHANNEL_PREFIX:
        needed = [prefix + "Z", prefix + "N", prefix + "E"]
        ok = 0
        for ch in needed:
            try:
                client.get_waveforms(net, sta, "*", ch, start, end)
                ok += 1
            except:
                pass
        if ok == 3:
            return prefix
    return None

# ============================================================
# 6. DOWNLOAD 3 KANAL
# ============================================================

def download_event(row):
    event_id = row['Event ID']
    origin = UTCDateTime(row['time_dt'])

    event_dir = os.path.join(OUTPUT_DIR, event_id)
    os.makedirs(event_dir, exist_ok=True)

    # resume
    if len(os.listdir(event_dir)) >= 3:
        return "skip"

    net, sta = pick_best_station(origin)
    if net is None or sta is None:
        return "fail"

    prefix = find_available_prefix(net, sta, origin)
    if prefix is None:
        return "fail"

    start = origin - WINDOW_BEFORE
    end = origin + WINDOW_AFTER

    success = 0
    for comp in ["Z", "N", "E"]:
        ch = prefix + comp
        out_file = os.path.join(event_dir, f"{ch}.mseed")
        try:
            st = client.get_waveforms(net, sta, "*", ch, start, end)
            st.write(out_file, format="MSEED")
            success += 1
        except:
            continue

    return "ok" if success == 3 else "fail"

# ============================================================
# 7. MULTI-THREAD + PROGRESS BAR
# ============================================================

results = {"ok": 0, "fail": 0, "skip": 0}

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(download_event, row) for _, row in iris.iterrows()]

    for future in tqdm(as_completed(futures), total=len(futures), desc="Downloading 3C Waveforms (SNR + Fallback)"):
        results[future.result()] += 1

print("\n=== RINGKASAN ===")
print(f"Berhasil   : {results['ok']}")
print(f"Gagal      : {results['fail']}")
print(f"Sudah ada  : {results['skip']}")
print("=== SELESAI ===")
